In [21]:
# !pip install faiss-cpu sentence-transformers PyPDF2 pandas ipywidgets
'''Author
Developer: Kajal Dadas
Email: kajaldadas149@gmail.com
This project was designed to demonstrate AI-driven recruitment automation using modern NLP techniques.'''

import os
import shutil
import re
import faiss
import numpy as np
import pandas as pd
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import ipywidgets as widgets
from IPython.display import display, HTML

# -------------------------
# FOLDERS
# -------------------------
UPLOAD_FOLDER = "uploaded_resumes"
SCREENED_FOLDER = "screened_resumes"

os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(SCREENED_FOLDER, exist_ok=True)

# -------------------------
# MODEL
# -------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

scoreboard_df = None


# -------------------------
# TEXT EXTRACTION
# -------------------------
def extract_text(pdf_path):

    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    return text.lower()


# -------------------------
# KEYWORD EXTRACTION FROM JD
# -------------------------
def extract_keywords(text):

    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())

    stopwords = {
        "with","and","the","for","are","you","will","have",
        "this","that","from","our","your","about","who",
        "their","them","into","such","also"
    }

    keywords = [w for w in words if w not in stopwords]

    return list(set(keywords))


# -------------------------
# HEADER
# -------------------------
display(HTML("""
<h2 style='text-align:center'>AI Resume Screening Agent</h2>
<p style='text-align:center;color:gray'>Strict Keyword + AI Matching</p>
"""))


# -------------------------
# JD PROMPT
# -------------------------
jd_box = widgets.Textarea(
    placeholder="Paste Job Description...",
    layout=widgets.Layout(width='100%', height='120px')
)

display(jd_box)


# -------------------------
# TOP N FIELD
# -------------------------
top_n_box = widgets.IntText(
    value=5,
    description="Top N:"
)

display(top_n_box)


# -------------------------
# FILE UPLOADER
# -------------------------
uploader = widgets.FileUpload(
    accept=".pdf",
    multiple=True
)

display(uploader)


# -------------------------
# BUTTONS
# -------------------------
upload_btn = widgets.Button(description="Upload Resume", button_style="success")
screen_btn = widgets.Button(description="Screen Resume", button_style="primary")
scoreboard_btn = widgets.Button(description="Check Scoreboard", button_style="info")

display(widgets.HBox([upload_btn, screen_btn, scoreboard_btn]))

output = widgets.Output()
display(output)


# -------------------------
# UPLOAD FUNCTION
# -------------------------
def upload_resumes(b):

    with output:
        output.clear_output()

        if len(uploader.value) == 0:
            print("Select resumes first")
            return

        for name, file_info in uploader.value.items():

            path = os.path.join(UPLOAD_FOLDER, name)

            with open(path, "wb") as f:
                f.write(file_info["content"])

        print("Resumes uploaded successfully")


upload_btn.on_click(upload_resumes)


# -------------------------
# GENERATE SCOREBOARD
# -------------------------
def generate_scoreboard():

    global scoreboard_df

    jd_text = jd_box.value.lower()

    jd_keywords = extract_keywords(jd_text)

    resumes = []
    names = []

    for file in os.listdir(UPLOAD_FOLDER):

        if file.endswith(".pdf"):

            path = os.path.join(UPLOAD_FOLDER, file)

            text = extract_text(path)

            resumes.append(text)
            names.append(file)

    if len(resumes) == 0:
        print("No resumes found")
        return

    # semantic embeddings
    resume_embeddings = model.encode(resumes)
    jd_embedding = model.encode([jd_text])

    dim = resume_embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)
    index.add(np.array(resume_embeddings))

    distances, indices = index.search(np.array(jd_embedding), len(names))

    results = []

    for rank, idx in enumerate(indices[0]):

        resume_text = resumes[idx]

        # keyword matching
        matches = sum(1 for k in jd_keywords if k in resume_text)

        keyword_ratio = matches / max(len(jd_keywords),1)

        semantic_distance = distances[0][rank]

        semantic_score = max(0, 100 - semantic_distance*10)

        # STRICT LOGIC
        if keyword_ratio < 0.05:
            score = min(semantic_score, 20)

        else:
            score = semantic_score * keyword_ratio

        results.append({
            "Rank": rank+1,
            "Resume": names[idx],
            "Keyword Matches": matches,
            "Match %": round(score,2)
        })

    scoreboard_df = pd.DataFrame(results).sort_values(
        by="Match %", ascending=False
    )


# -------------------------
# SCREEN FUNCTION
# -------------------------
def screen_resumes(b):

    with output:
        output.clear_output()

        generate_scoreboard()

        if scoreboard_df is None:
            return

        top_n = top_n_box.value

        top_resumes = scoreboard_df.head(top_n)

        for resume in top_resumes["Resume"]:

            shutil.move(
                os.path.join(UPLOAD_FOLDER, resume),
                os.path.join(SCREENED_FOLDER, resume)
            )

        print(f"Top {top_n} resumes moved to screened folder")


screen_btn.on_click(screen_resumes)


# -------------------------
# SHOW SCOREBOARD
# -------------------------
def show_scoreboard(b):

    with output:
        output.clear_output()

        generate_scoreboard()

        if scoreboard_df is None:
            return

        display(scoreboard_df)


scoreboard_btn.on_click(show_scoreboard)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Textarea(value='', layout=Layout(height='120px', width='100%'), placeholder='Paste Job Description...')

IntText(value=5, description='Top N:')

FileUpload(value={}, accept='.pdf', description='Upload', multiple=True)

Output()